In [2]:
from pathlib import Path

DATA_DIR = Path("..") / "data"

for f in DATA_DIR.rglob("*"):
    if f.is_file():
        print(f.name, round(f.stat().st_size / 1024**3, 2), "GB")

digikala-comments.csv 1.19 GB
digikala-products.csv 0.24 GB


In [3]:
import duckdb

con = duckdb.connect()
COMMENTS = "../data/digikala-comments.csv"

In [4]:
con.sql(f"""
    DESCRIBE
    SELECT * FROM read_csv('{COMMENTS}', all_varchar=true)
""").show()

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ title                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ body                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ created_at            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rate                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ recommendation_status │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_buyer              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ advantages            │ VARCHAR     │ 

In [5]:
n_rows = con.sql(f"""
    SELECT COUNT(*) FROM read_csv('{COMMENTS}', all_varchar=true)
""").fetchall()[0][0]

print(f"{n_rows:,}")

6,156,289


In [11]:
con.sql(f"""
    SELECT
        COUNT(*)                    AS n_rows,
        COUNT(DISTINCT id)          AS unique_ids,
        COUNT(DISTINCT product_id)  AS unique_products,
        COUNT(DISTINCT seller_code) AS unique_sellers
    FROM read_csv('{COMMENTS}', all_varchar=true)
""").show()

┌─────────┬────────────┬─────────────────┬────────────────┐
│ n_rows  │ unique_ids │ unique_products │ unique_sellers │
│  int64  │   int64    │      int64      │     int64      │
├─────────┼────────────┼─────────────────┼────────────────┤
│ 6156289 │    6153060 │          331599 │          29214 │
└─────────┴────────────┴─────────────────┴────────────────┘



In [7]:
con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE id IS NULL)     AS null_ids,
        COUNT(*) FILTER (WHERE id IS NOT NULL) AS non_null_ids,
        COUNT(DISTINCT id)                     AS unique_ids
    FROM read_csv('{COMMENTS}', all_varchar=true)
""").show()

┌──────────┬──────────────┬────────────┐
│ null_ids │ non_null_ids │ unique_ids │
│  int64   │    int64     │   int64    │
├──────────┼──────────────┼────────────┤
│        0 │      6156289 │    6153060 │
└──────────┴──────────────┴────────────┘



In [8]:
con.sql(f"""
    WITH per_product AS (
        SELECT product_id, COUNT(*) AS n_reviews
        FROM read_csv('{COMMENTS}', all_varchar=true)
        WHERE product_id IS NOT NULL
        GROUP BY product_id
    )
    SELECT
        COUNT(*)                       AS n_products,
        MIN(n_reviews)                 AS min_reviews,
        quantile_cont(n_reviews, 0.5)  AS median_reviews,
        AVG(n_reviews)                 AS mean_reviews,
        quantile_cont(n_reviews, 0.99) AS p99_reviews,
        MAX(n_reviews)                 AS max_reviews
    FROM per_product
""").show()

┌────────────┬─────────────┬────────────────┬────────────────────┬─────────────┬─────────────┐
│ n_products │ min_reviews │ median_reviews │    mean_reviews    │ p99_reviews │ max_reviews │
│   int64    │    int64    │     double     │       double       │   double    │    int64    │
├────────────┼─────────────┼────────────────┼────────────────────┼─────────────┼─────────────┤
│     331599 │           1 │            4.0 │ 18.565463104532885 │       200.0 │         400 │
└────────────┴─────────────┴────────────────┴────────────────────┴─────────────┴─────────────┘



## A1 Findings

- The comments file has 6,156,289 records and 15 columns (1.19 GB), counted with a CSV parser.
- No record has a missing id, but there are only 6,153,060 distinct ids, so 3,229 records reuse an id seen earlier (to be investigated).
- The data covers 331,599 distinct products and 29,214 distinct seller codes.
- Reviews per product are highly skewed: min 1, median 4, mean 18.6, 99th percentile 200, max 400.
- Implication: the largest product holds under 0.01% of all records, so a product-grouped split can reach the 70/15/15 target closely.

In [9]:
con.sql(f"""
    CREATE OR REPLACE TEMP VIEW comments AS
    SELECT * FROM read_csv('{COMMENTS}', all_varchar=true)
""")

CountBIGINT


In [10]:
con.sql("""
    CREATE OR REPLACE TEMP TABLE dup_rows AS
    SELECT * FROM comments
    WHERE id IN (
        SELECT id FROM comments GROUP BY id HAVING COUNT(*) > 1
    )
""")

CountBIGINT
6455


In [11]:
con.sql("""
    SELECT
        CASE WHEN n_versions = 1 THEN 'identical copies'
             ELSE 'same id, different content' END AS kind,
        COUNT(*)    AS n_ids,
        SUM(n_rows) AS n_rows
    FROM (
        SELECT id,
               COUNT(*) AS n_rows,
               COUNT(DISTINCT md5(concat_ws('|', title, body, created_at, rate,
                    recommendation_status, product_id, seller_code))) AS n_versions
        FROM dup_rows
        GROUP BY id
    )
    GROUP BY kind
""").show()

┌────────────────────────────┬───────┬────────┐
│            kind            │ n_ids │ n_rows │
│          varchar           │ int64 │ int128 │
├────────────────────────────┼───────┼────────┤
│ same id, different content │   387 │    774 │
│ identical copies           │  2839 │   5681 │
└────────────────────────────┴───────┴────────┘



In [12]:
con.sql("""
    SELECT id, product_id, rate, left(body, 60) AS body_start
    FROM dup_rows
    ORDER BY id
    LIMIT 10
""").show()

┌──────────┬────────────┬─────────┬──────────────────────────────────────────────────────────────┐
│    id    │ product_id │  rate   │                          body_start                          │
│ varchar  │  varchar   │ varchar │                           varchar                            │
├──────────┼────────────┼─────────┼──────────────────────────────────────────────────────────────┤
│ 10021106 │ 252058     │ 0.0     │ متن روان و گویندگی با صدای دلنشین و بسیار اموزنده            │
│ 10021106 │ 252058     │ 0.0     │ متن روان و گویندگی با صدای دلنشین و بسیار اموزنده            │
│ 1002211  │ 65742      │ 3.0     │ یکی از قشنگترین داستانهای کودکیم، پیشنهاد میکنم حتما گوش بدی │
│ 1002211  │ 65742      │ 3.0     │ یکی از قشنگترین داستانهای کودکیم، پیشنهاد میکنم حتما گوش بدی │
│ 10051223 │ 252058     │ 3.0     │ افزایش سطح اگاهی وبه عنوان هدیه مناسب می باشد.ممنون          │
│ 10051223 │ 252058     │ 3.0     │ افزایش سطح اگاهی وبه عنوان هدیه مناسب می باشد.ممنون          │
│ 10115646

In [13]:
con.sql("""
    WITH per_product AS (
        SELECT product_id, COUNT(*) AS n_reviews
        FROM comments
        WHERE product_id IS NOT NULL
        GROUP BY product_id
    )
    SELECT n_reviews, COUNT(*) AS n_products
    FROM per_product
    WHERE n_reviews IN (198, 199, 200, 201, 202, 398, 399, 400)
    GROUP BY n_reviews
    ORDER BY n_reviews
""").show()

┌───────────┬────────────┐
│ n_reviews │ n_products │
│   int64   │   int64    │
├───────────┼────────────┤
│       198 │         34 │
│       199 │         47 │
│       200 │       8400 │
│       201 │        542 │
│       202 │        297 │
│       400 │          3 │
└───────────┴────────────┘



In [14]:
cols = [r[0] for r in con.sql("DESCRIBE comments").fetchall()]

parts = [
    f"COUNT(*) FILTER (WHERE \"{c}\" IS NULL OR trim(\"{c}\") = '') AS \"{c}\""
    for c in cols
]

miss = con.sql(
    "SELECT COUNT(*) AS total_rows, " + ", ".join(parts) + " FROM comments"
).df()

t = miss.T
t.columns = ["missing"]
total = int(t.loc["total_rows", "missing"])
t = t.drop("total_rows")
t["missing_pct"] = (t["missing"] / total * 100).round(2)
t.sort_values("missing_pct", ascending=False)

,missing,missing_pct
title,1353,0.02
id,0,0.00
body,0,0.00
created_at,0,0.00
rate,0,0.00
recommendation_status,0,0.00
is_buyer,0,0.00
product_id,0,0.00
advantages,0,0.00
disadvantages,0,0.00


In [15]:
con.sql("""
    SELECT recommendation_status, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM comments
    GROUP BY recommendation_status
    ORDER BY n DESC
""").show()

┌───────────────────────┬─────────┬────────┐
│ recommendation_status │    n    │  pct   │
│        varchar        │  int64  │ double │
├───────────────────────┼─────────┼────────┤
│ recommended           │ 4162839 │  67.62 │
│ nan                   │  894426 │  14.53 │
│ no_idea               │  594121 │   9.65 │
│ not_recommended       │  504903 │    8.2 │
└───────────────────────┴─────────┴────────┘



In [16]:
con.sql("""
    SELECT is_buyer, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) FILTER (WHERE recommendation_status IS NULL)
                 / COUNT(*), 2) AS pct_missing_label
    FROM comments
    GROUP BY is_buyer
    ORDER BY n DESC
""").show()

┌──────────┬─────────┬───────────────────┐
│ is_buyer │    n    │ pct_missing_label │
│ varchar  │  int64  │      double       │
├──────────┼─────────┼───────────────────┤
│ True     │ 5854108 │               0.0 │
│ False    │  302181 │               0.0 │
└──────────┴─────────┴───────────────────┘



In [17]:
con.sql("""
    SELECT rate, COUNT(*) AS n
    FROM comments
    GROUP BY rate
    ORDER BY n DESC
    LIMIT 20
""").show()

┌─────────┬─────────┐
│  rate   │    n    │
│ varchar │  int64  │
├─────────┼─────────┤
│ 5.0     │ 2398801 │
│ 4.0     │ 1515723 │
│ 3.0     │  958799 │
│ 0.0     │  529098 │
│ 1.0     │  327662 │
│ 2.0     │  198077 │
│ 4.5     │   13479 │
│ 3.5     │   12207 │
│ 4.25    │   11653 │
│ 3.75    │   10045 │
│ 4.2     │    9843 │
│ 3.8     │    8745 │
│ 4.4     │    8675 │
│ 4.75    │    8292 │
│ 4.6     │    8152 │
│ 3.25    │    7695 │
│ 4.8     │    7276 │
│ 3.4     │    7250 │
│ 3.6     │    6975 │
│ 2.5     │    5438 │
└─────────┴─────────┘
       20 rows     

